In [ ]:
from prc_fuel.features import duration_s
from prc_fuel.load import load, q

In [ ]:
# trajectory summary per interval, one flight
q("""
SELECT
  f.idx,
  COUNT(ft.flight_id) AS n_points,
  AVG(ft.altitude) AS avg_altitude,
  MIN_BY(ft.altitude, IF(ft.altitude IS NULL, NULL, ft.`timestamp`)) AS alt_start,
  MAX_BY(ft.altitude, IF(ft.altitude IS NULL, NULL, ft.`timestamp`)) AS alt_end,
  AVG(ft.groundspeed) AS avg_groundspeed,
  AVG(ft.vertical_rate) AS avg_vertical_rate,
  AVG(ft.mach) AS avg_mach
FROM `prc-data-challenge-2025.prc2025.fuel_train` AS f
LEFT JOIN `prc-data-challenge-2025.prc2025.flights_train` AS ft
  ON ft.flight_id = f.flight_id
  AND ft.`timestamp` BETWEEN f.start AND f.`end`
WHERE f.flight_id = 'prc770822360'
GROUP BY f.idx
ORDER BY f.idx
""")

Intervals 0 to 3 are cruise, ~36000 ft and vertical rate close to 0. Interval 4 is the descent, 36000 -> 15225 ft. The mean altitude alone (29500) didn't show how much it went down, hence alt_start / alt_end.
Units: ft, kt, ft/min.

In [ ]:
# points sitting exactly on the interval bounds
q("""
SELECT
  f.idx,
  ft.`timestamp` = f.start AS at_start,
  ft.altitude,
  ft.groundspeed,
  ft.mach,
  ft.source
FROM `prc-data-challenge-2025.prc2025.fuel_train` AS f
JOIN `prc-data-challenge-2025.prc2025.flights_train` AS ft
  ON ft.flight_id = f.flight_id
  AND (ft.`timestamp` = f.start OR ft.`timestamp` = f.`end`)
WHERE f.flight_id = 'prc770822360'
ORDER BY f.idx, at_start DESC
""")

On this flight every interval has a point exactly on start and on end. They come from acars, have a mach but no altitude or groundspeed. So a plain MIN_BY(altitude, timestamp) picks them and returns null, that's why the IF skips points without altitude.
Bonus: mach is there, so we get some airspeed after all. (On most other flights these acars points do have an altitude, see below.)

In [ ]:
# same summary on all flights, only the row count (~4.7 GB read)
q("""
SELECT COUNT(*) AS n_rows
FROM (
  SELECT
    f.flight_id,
    f.idx,
    COUNT(ft.flight_id) AS n_points,
    AVG(ft.altitude) AS avg_altitude,
    MIN_BY(ft.altitude, IF(ft.altitude IS NULL, NULL, ft.`timestamp`)) AS alt_start,
    MAX_BY(ft.altitude, IF(ft.altitude IS NULL, NULL, ft.`timestamp`)) AS alt_end,
    AVG(ft.groundspeed) AS avg_groundspeed,
    AVG(ft.vertical_rate) AS avg_vertical_rate,
    AVG(ft.mach) AS avg_mach
  FROM `prc-data-challenge-2025.prc2025.fuel_train` AS f
  LEFT JOIN `prc-data-challenge-2025.prc2025.flights_train` AS ft
    ON ft.flight_id = f.flight_id
    AND ft.`timestamp` BETWEEN f.start AND f.`end`
  GROUP BY f.flight_id, f.idx
)
""")

131530 rows, same as fuel_train, so no interval lost or duplicated by the join. Took ~5 s in BigQuery Studio for 4.7 GB read.

In [ ]:
df = load("train")

traj_cols = [
    "n_points",
    "avg_altitude",
    "alt_start",
    "alt_end",
    "avg_groundspeed",
    "avg_vertical_rate",
    "avg_mach",
]
traj = df[traj_cols]

traj.isna().sum()

Missing per column: altitude ~3% (4024), groundspeed and vertical rate ~35%, mach ~36%. Three groups, the altitudes always missing together and the two speeds too.

In [ ]:
traj[["avg_altitude", "avg_groundspeed", "avg_mach"]].isna().value_counts()

Groundspeed and mach are not missing on the same intervals. At least one speed in ~83% of intervals (46 + 19 + 17 + 1 %), only altitude in 15%, nothing at all in 2%.
mach is airspeed and groundspeed is speed over ground, the difference is the wind. Could convert mach to a speed later to fill the gaps.

In [ ]:
traj.describe()

n_points: 25% of intervals have 2 points or less, probably just the two acars points on the bounds. Looks like the ~40% "without trajectory" from the exploration.

Median altitude ~37000 ft and groundspeed ~440 kt, mostly cruise as expected. Some values are impossible though:
- alt_start / alt_end go from -14200 to 131037 ft, but avg_altitude stays between -400 and 43000. So it's single bad points on the bounds, the mean smooths them out.
- avg_vertical_rate from -8754 to 11905 ft/min, way too much as an average over minutes.
- groundspeed up to 975 kt, far above the ~440 kt median.
- mach = 0 on a flying plane.
Need to see how many there are before deciding what to do (set to NaN probably).

In [ ]:
(traj.avg_mach == 0).sum()

63 intervals with avg_mach exactly 0 (~0.07% of those with a mach). Either ground or a sensor sending 0 instead of null, not a real airspeed. Few enough to set to NaN.

In [ ]:
traj.avg_groundspeed.nlargest(20)

Values from ~595 to 632 kt follow each other closely, often consecutive intervals of the same flight, so real strong tailwinds. Then a 50 kt jump to 682 and a 293 kt jump to 975. The 975 is clearly isolated, the 682 needs a look at its neighbours.

In [ ]:
df.loc[30780:30788, ["flight_id", "idx", "avg_groundspeed", "avg_altitude", "n_points"]]

The 682 kt interval is the first of its flight, the next ones are 573 and 577 kt then down to ~470, all at 38000 ft with 566 points. A strong jet stream fading, so real. Only the 975 kt one is an error. Threshold 800 kt, in the gap between 682 and 975.

In [ ]:
gs = traj.avg_groundspeed
q1, q3 = gs.quantile(0.25), gs.quantile(0.75)
q3 + 3 * (q3 - q1)

Tukey far out upper bound is 656 kt, it would drop the real 682 kt interval. Tukey can't work on the low side either: climbs and descents sit far below the cruise block. The gap in the sorted values, checked on the neighbours, is more reliable here. Threshold kept at 800 kt (1 interval).

In [ ]:
traj.alt_start.nlargest(20)

A tight block at ~43000 ft (FL430, real cruise level), then a 4400 ft jump to an isolated 47400 and a 33000 ft jump to 80144, clearly an error. The 131037 max seen in describe comes from alt_end. 47400 needs a look at its neighbours.

In [ ]:
df.loc[
    119619:119627,
    [
        "flight_id",
        "idx",
        "aircraft_type",
        "alt_start",
        "alt_end",
        "avg_altitude",
        "n_points",
    ],
]

The 47400 is a single point on the boundary between two intervals of an A359 landing (2676 ft before, ground after), so it shows up as alt_end of one and alt_start of the next. Averages are fine. The flight below has 2 points per interval at exactly 39000 ft, one of the intervals without trajectory.

In [ ]:
traj.alt_start.nsmallest(20)

A block of repeated values at -375 / -400 ft (ADS-B steps of 25 ft), a plane on the ground with high pressure, real. Then isolated -1100 and -1500, and -10300 clearly an error. -1100 needs a look at its neighbours.

In [ ]:
df.loc[
    64440:64448,
    ["flight_id", "idx", "alt_start", "alt_end", "avg_altitude", "n_points"],
]

The -1100 is the boundary point of a landing, the next interval is on the ground at ~-110 ft, so it's ~1000 ft below the runway, an error. Lower threshold -1000 ft, in the gap between the -400 block and -1100. No need to check -1500, nothing real sits between -400 and -1100.

In [ ]:
traj.alt_end.nlargest(20)

alt_end has more values above the ~43000 block than alt_start, spread from 44700 to 49900 with no clear gap, all rounded to the hundred unlike the real ones (43002, 43003...). Probably boundary points from another source. Checking 44700, the lowest of them.

In [ ]:
df.loc[
    63001:63009,
    [
        "flight_id",
        "idx",
        "aircraft_type",
        "alt_start",
        "alt_end",
        "avg_altitude",
        "n_points",
    ],
]

44700 is the alt_end of the last interval of an A20N landing (4355 ft at start, 2533 ft average), so an error. Bad points at the end of a flight only show up in alt_end, which is why it has more of them. Upper threshold 44000 ft, between the real ~43000 block and 44700, also fine for alt_start.

In [ ]:
traj.alt_end.nsmallest(20)

Real values go down to -450 ft in 25 ft steps, below that only round values (-1100, -1500, -4700, -14200), same pattern as on the high side. -1100 and -1500 are the boundary points already seen in alt_start.

Altitude thresholds: -1000 to 44000 ft for alt_start and alt_end. avg_altitude stays within -400 to 43000, no threshold needed.

In [ ]:
traj.avg_vertical_rate.nlargest(20)

No clear gap: values follow each other from ~3600 to 5100 ft/min, then 6956, 7337, 7778 and an isolated 11905. Most of the top ones are consecutive intervals of the same flights. Checking against the altitude change over the interval.

In [ ]:
df.assign(duration_min=duration_s(df) / 60).loc[
    16687:16691,
    [
        "flight_id",
        "alt_start",
        "alt_end",
        "avg_altitude",
        "avg_vertical_rate",
        "duration_min",
        "n_points",
    ],
]

The 11905 ft/min interval is flat cruise (36001 -> 36000 ft in 4 min). The whole flight is off, not one point: 1322 ft/min over 56 min at constant altitude, 4651 for a 1000 ft step in 4 min (250 ft/min). A threshold would only catch the extreme one.

In [ ]:
check = df.assign(implied_vr=(df.alt_end - df.alt_start) / (duration_s(df) / 60))
check[["implied_vr", "avg_vertical_rate"]].corr()

implied_vr is the rate implied by the altitude change, a mean over time. Correlation with avg_vertical_rate is only 0.41. Could be dragged down by the bad boundary altitudes, redoing it with valid ones.

In [ ]:
ok = check.alt_start.between(-1000, 44000) & check.alt_end.between(-1000, 44000)
check.loc[ok, ["implied_vr", "avg_vertical_rate"]].corr()

Still 0.42, so the bad altitudes weren't the cause. avg_vertical_rate is a mean per point over the points that have one (35% missing), not over time, and some flights are plain wrong. No threshold for it: implied_vr will be tried as a feature and both compared in runs.

In [ ]:
traj.avg_mach.nlargest(20)

Max avg_mach is 0.87, values follow each other closely, often consecutive intervals of the same flight holding cruise mach. No upper threshold needed.

In [ ]:
traj.avg_mach[traj.avg_mach > 0].nsmallest(20)

No gap above 0: the next values are a block at exactly 0.05 (~33 kt, taxi or start of takeoff roll), probably rounded by the source. Checking whether the mach 0 intervals are on the ground or in flight.

In [ ]:
traj.loc[traj.avg_mach == 0, "avg_altitude"].describe()

Median altitude of the mach 0 intervals is 3866 ft, 75% above 2026 ft, max 40949 ft. Most are in flight, so it's the source sending 0 instead of null. All 63 set to NaN.

## Thresholds

All taken from gaps in the data and checked on neighbouring intervals:

| column | rule | why |
|---|---|---|
| alt_start, alt_end | NaN outside -1000 to 44000 ft | real values from -450 to 43006, beyond that isolated round values |
| avg_altitude | none | stays within -400 to 43000 |
| avg_groundspeed | NaN above 800 kt | gap between 682 (real jet stream) and 975 |
| avg_mach | NaN when 0 | mostly in flight, high side clean (max 0.87) |
| avg_vertical_rate | none | doesn't match the altitude change (corr 0.42), implied_vr tried instead |

In [ ]:
from prc_fuel.clean import mask_bad_values

masked = mask_bad_values(df)
cols = ["alt_start", "alt_end", "avg_groundspeed", "avg_mach"]
masked[cols].isna().sum() - df[cols].isna().sum()

mask_bad_values adds 5 NaN to alt_start, 17 to alt_end (13 above 44000, 4 below -1000), 1 to avg_groundspeed and 63 to avg_mach, exactly the values found above.